# 3: Generate Embeddings & Upload to Pinecone
## RAG-Based Medical QA System — PubMedQA
---
**What this notebook does:**
- Loads recursive chunks (our main strategy)
- Generates embeddings using all-MiniLM-L6-v2 (384 dimensions)
- Uploads all embeddings + metadata to Pinecone in batches
- Verifies the upload was successful
- Also pre-computes BM25 index and saves it for Step 4

**Runtime:** Enable GPU in Colab → Runtime > Change runtime type > T4 GPU


## 3.1 Install Libraries

In [ ]:
!pip install sentence-transformers pinecone rank-bm25 tqdm pandas -q
print('All libraries installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 25.1 MB/s eta 0:00:00
All libraries installed!


## 3.2 Imports & API Keys
⚠️ Fill in your API keys below before running!

In [ ]:
import json
import time
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from rank_bm25 import BM25Okapi

# ============================================================
#  FILL IN YOUR KEYS HERE
# ============================================================
PINECONE_API_KEY = 'pcsk_4EURsR_7vhX2ipcBnChK6GbiyWKXFFWTgXjCTezhT1MuZufSHd5ehrHNdJQsRjHuBrvuha'
INDEX_NAME       = 'pubmed-rag'
# ============================================================

print('Imports done!')

Imports done!


## 3.3 Upload & Load Chunks

In [ ]:
from google.colab import files
print('Please upload chunks_recursive.json')
uploaded = files.upload()

Please upload chunks_recursive.json


Saving chunks_recursive.json to chunks_recursive.json


In [ ]:
with open('chunks_recursive.json', 'r', encoding='utf-8') as f:
    recursive_chunks = json.load(f)

print(f'Loaded {len(recursive_chunks)} recursive chunks')
print(f'Sample chunk keys: {list(recursive_chunks[0].keys())}')
print(f'Sample text: {recursive_chunks[0]["text"][:150]}...')

Loaded 11605 recursive chunks
Sample chunk keys: ['chunk_id', 'doc_id', 'pubid', 'chunk_index', 'text', 'word_count', 'char_count', 'strategy', 'source_question']
Sample text: Although the use of alternative medicine in the United States is increasing, no published studies have documented the effectiveness of naturopathy for...


## 3.4 Load Embedding Model
We use `all-MiniLM-L6-v2` — lightweight, fast, 384-dim output.
This matches the Pinecone index dimension exactly.

In [ ]:
print('Loading embedding model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Verify dimensions with a test
test_emb = embedder.encode(['test sentence'])
print(f'Model loaded! Embedding dimension: {test_emb.shape[1]}')
assert test_emb.shape[1] == 384, 'Dimension mismatch! Check Pinecone index settings.'
print('Dimension check passed ✅')

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded! Embedding dimension: 384
Dimension check passed ✅


## 3.5 Generate Embeddings in Batches
We embed all 11,604 chunks in batches of 256 for speed.

In [ ]:
def generate_embeddings_batched(chunks, embedder, batch_size=256):
    """
    Generate embeddings for all chunks in batches.
    Returns numpy array of shape (num_chunks, 384)
    """
    all_texts = [c['text'] for c in chunks]
    all_embeddings = []

    for i in tqdm(range(0, len(all_texts), batch_size), desc='Generating embeddings'):
        batch = all_texts[i:i + batch_size]
        batch_embs = embedder.encode(
            batch,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True  # normalize for cosine similarity
        )
        all_embeddings.append(batch_embs)

    return np.vstack(all_embeddings)


print('Generating embeddings (this takes 3-5 mins with GPU)...')
start_time = time.time()
embeddings = generate_embeddings_batched(recursive_chunks, embedder, batch_size=256)
elapsed = time.time() - start_time

print(f'\nEmbeddings generated!')
print(f'Shape     : {embeddings.shape}')   # should be (11604, 384)
print(f'Time taken: {elapsed:.1f} seconds')
print(f'Dtype     : {embeddings.dtype}')

Generating embeddings (this takes 3-5 mins with GPU)...


Generating embeddings: 100%|██████████| 46/46 [00:24<00:00,  1.84it/s]


Embeddings generated!
Shape     : (11605, 384)
Time taken: 25.0 seconds
Dtype     : float32


## 3.6 Connect to Pinecone & Verify Index

In [ ]:
# Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

# Check index exists
existing_indexes = [idx.name for idx in pc.list_indexes()]
print(f'Existing indexes: {existing_indexes}')

if INDEX_NAME not in existing_indexes:
    print(f'ERROR: Index "{INDEX_NAME}" not found!')
    print('Please create it at pinecone.io with dimension=384, metric=cosine')
else:
    print(f'Index "{INDEX_NAME}" found ✅')

# Connect to index
index = pc.Index(INDEX_NAME)

# Check current stats
stats = index.describe_index_stats()
print(f'\nIndex stats before upload:')
print(f'  Total vectors: {stats.total_vector_count}')
print(f'  Dimension    : {stats.dimension}')

Existing indexes: ['pubmed-rag']
Index "pubmed-rag" found ✅

Index stats before upload:
  Total vectors: 0
  Dimension    : 384


## 3.7 Upload Embeddings to Pinecone
We upload in batches of 100 vectors (Pinecone's recommended batch size).
Each vector includes metadata for retrieval later.

In [ ]:
def upload_to_pinecone(chunks, embeddings, index, batch_size=100):
    """
    Upload chunk embeddings + metadata to Pinecone.

    Each vector stored as:
      id       → chunk_id
      values   → 384-dim embedding
      metadata → text, doc_id, pubid, word_count, source_question
    """
    total_uploaded = 0

    for i in tqdm(range(0, len(chunks), batch_size), desc='Uploading to Pinecone'):
        batch_chunks = chunks[i:i + batch_size]
        batch_embs   = embeddings[i:i + batch_size]

        vectors = []
        for chunk, emb in zip(batch_chunks, batch_embs):
            vectors.append({
                'id'    : chunk['chunk_id'],
                'values': emb.tolist(),
                'metadata': {
                    'text'            : chunk['text'][:1000],  # Pinecone metadata limit
                    'doc_id'          : chunk['doc_id'],
                    'pubid'           : chunk['pubid'],
                    'word_count'      : chunk['word_count'],
                    'source_question' : chunk['source_question'][:200],
                    'chunk_index'     : chunk['chunk_index'],
                    'strategy'        : chunk['strategy']
                }
            })

        # Upsert batch
        index.upsert(vectors=vectors)
        total_uploaded += len(vectors)

        # Small delay to avoid rate limiting
        time.sleep(0.05)

    return total_uploaded


print('Uploading to Pinecone (this takes 5-8 mins)...')
start_time = time.time()
total = upload_to_pinecone(recursive_chunks, embeddings, index, batch_size=100)
elapsed = time.time() - start_time

print(f'\nUpload complete!')
print(f'Total vectors uploaded : {total}')
print(f'Time taken            : {elapsed:.1f} seconds')

Uploading to Pinecone (this takes 5-8 mins)...


Uploading to Pinecone: 100%|██████████| 117/117 [01:22<00:00,  1.42it/s]


Upload complete!
Total vectors uploaded : 11605
Time taken            : 82.7 seconds


## 3.8 Verify Upload in Pinecone

In [ ]:
# Wait a moment for Pinecone to index everything
print('Waiting 10 seconds for Pinecone to index...')
time.sleep(10)

# Check stats after upload
stats = index.describe_index_stats()
print(f'\nIndex stats AFTER upload:')
print(f'  Total vectors: {stats.total_vector_count}')
print(f'  Dimension    : {stats.dimension}')

# Do a quick test query
print('\nRunning test query...')
test_query = 'What are the effects of aspirin on cardiovascular disease?'
test_emb   = embedder.encode([test_query], normalize_embeddings=True)[0].tolist()

results = index.query(
    vector=test_emb,
    top_k=3,
    include_metadata=True
)

print(f'\nTest query: "{test_query}"')
print(f'Top 3 results:')
for i, match in enumerate(results['matches']):
    print(f'\n  [{i+1}] Score: {match["score"]:.4f}')
    print(f'       ID   : {match["id"]}')
    print(f'       Text : {match["metadata"]["text"][:150]}...')

Waiting 10 seconds for Pinecone to index...

Index stats AFTER upload:
  Total vectors: 11605
  Dimension    : 384

Running test query...

Test query: "What are the effects of aspirin on cardiovascular disease?"
Top 3 results:

  [1] Score: 0.6346
       ID   : recursive_010848
       Text : Since the 1980s, clinical trial evidence has supported aspirin use in the secondary prevention of cardiovascular disease (CVD).AIM: To explore aspirin...

  [2] Score: 0.6257
       ID   : recursive_010849
       Text : . Overall, 547 men (9.5%) were taking aspirin daily, of whom 321 (59%) had documented CVD. Among men with pre-existing disease, 153 out of 345 (44%) m...

  [3] Score: 0.5558
       ID   : recursive_008567
       Text : . Of all 6588 patients investigated (women 25%), 4295 (65%) had no diabetes, 752 (11%) had incident diabetes and 1541 (23%) had prevalent diabetes. Al...


## 3.9 Build & Save BM25 Index
BM25 is the keyword-based retriever for our Hybrid Search.
We build it now and save it — it will be loaded in the main app.

In [ ]:
def build_bm25_index(chunks):
    """
    Build BM25 index over all chunk texts.
    Tokenizes by whitespace (simple but effective for medical text).
    """
    print('Tokenizing chunks for BM25...')
    tokenized_corpus = [
        chunk['text'].lower().split()
        for chunk in tqdm(chunks, desc='Tokenizing')
    ]

    print('Building BM25 index...')
    bm25 = BM25Okapi(tokenized_corpus)
    print(f'BM25 index built over {len(tokenized_corpus)} chunks ✅')
    return bm25, tokenized_corpus


bm25_index, tokenized_corpus = build_bm25_index(recursive_chunks)

# Save BM25 index + chunk texts for the app
bm25_data = {
    'bm25'            : bm25_index,
    'chunks'          : recursive_chunks,
    'tokenized_corpus': tokenized_corpus
}

with open('bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25_data, f)

print('BM25 index saved → bm25_index.pkl')

Tokenizing chunks for BM25...


Tokenizing: 100%|██████████| 11605/11605 [00:00<00:00, 98180.20it/s]

Building BM25 index...


BM25 index built over 11605 chunks ✅
BM25 index saved → bm25_index.pkl


## 3.10 Download BM25 Index

In [ ]:
from google.colab import files
files.download('bm25_index.pkl')
print('\nStep 3 Complete! ✅')
print('Files to keep safe:')
print('  bm25_index.pkl  → BM25 retriever (for hybrid search)')
print('  Pinecone index  → Semantic retriever (cloud, no download needed)')
print('\nNext: Step 4 — Build Hybrid Search + Re-Ranking Pipeline')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Step 3 Complete! ✅
Files to keep safe:
  bm25_index.pkl  → BM25 retriever (for hybrid search)
  Pinecone index  → Semantic retriever (cloud, no download needed)

Next: Step 4 — Build Hybrid Search + Re-Ranking Pipeline
